# 03 - Exploratory Data Analysis: El Clasico All-Time

**Dataset:** 202 La Liga El Clasico matches (1929-2026), 68 features

---

## Table of Contents

| # | Analysis | Chart Type |
|---|---------|------------|
| 1 | Dataset Overview | Table |
| 2 | All-Time Win Record: RM vs Barca vs Draw | Pie Chart |
| 3 | Goals Per Match Over Decades | Grouped Bar Chart |
| 4 | Total Goals Distribution | Histogram + KDE |
| 5 | Home vs Away Goal Comparison | Box Plot |
| 6 | RM Goals vs Barca Goals (Scatter) | Scatter Plot |
| 7 | Win Rate by Decade (Stacked) | Stacked Bar Chart |
| 8 | Home Advantage Analysis | Grouped Bar Chart |
| 9 | Monthly Match Distribution | Bar Chart |
| 10 | Shots vs Goals Correlation | Scatter + Regression |
| 11 | Fouls and Cards Per Match Over Time | Line Chart |
| 12 | Goal Category Breakdown | Donut Chart |
| 13 | Biggest Wins (Top 10) | Horizontal Bar Chart |
| 14 | Heatmap: Correlation Matrix | Heatmap |
| 15 | Corners Comparison | Violin Plot |
| 16 | Half-Time vs Full-Time Results | Grouped Bar |
| 17 | Shots on Target Efficiency | Bar Chart |
| 18 | Outlier Matches Visualization | Strip Plot |
| 19 | Cumulative Wins Over Time | Line Chart |
| 20 | K-Means Cluster Visualization | Scatter Plot |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Team colors
RM_COLOR = '#FEBE10'   # Real Madrid gold
BARCA_COLOR = '#A50044' # Barcelona maroon
DRAW_COLOR = '#808080'  # Gray for draws

# Load data
df = pd.read_csv('../data/final/el_clasico_final.csv')
print(f'Loaded: {df.shape[0]} matches, {df.shape[1]} features')
print(f'Date range: {df["year"].min():.0f} - {df["year"].max():.0f}')

---
## 1. Dataset Overview

In [ ]:
print(f'Total matches   : {len(df)}')
print(f'Date range      : {df["year"].min():.0f} - {df["year"].max():.0f}')
print(f'Total goals     : {df["total_goals"].sum():.0f}')
print(f'Avg goals/match : {df["total_goals"].mean():.2f}')
print(f'RM total goals  : {df["rm_goals"].sum():.0f}')
print(f'Barca total goals: {df["barca_goals"].sum():.0f}')
print()

results = df['rm_result'].value_counts()
print('Results from Real Madrid perspective:')
for res, count in results.items():
    print(f'  {res:10s}: {count} ({count/len(df)*100:.1f}%)')

print()
df.describe().round(2)

---
## 2. All-Time Win Record (Pie Chart)

In [ ]:
results = df['rm_result'].value_counts()
labels = results.index.tolist()
sizes = results.values

color_map = {'RM Win': RM_COLOR, 'Barca Win': BARCA_COLOR,
             'RM Loss': BARCA_COLOR, 'Draw': DRAW_COLOR}
colors = [color_map.get(l, '#999999') for l in labels]

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, autopct='%1.1f%%',
    startangle=90, textprops={'fontsize': 13, 'fontweight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for t in autotexts:
    t.set_color('white')
ax.set_title('El Clasico All-Time Results (Real Madrid Perspective)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Goals Per Match Over Decades (Grouped Bar)

In [ ]:
decade_df = df.dropna(subset=['decade']).copy()
decade_stats = decade_df.groupby('decade').agg(
    rm_avg=('rm_goals', 'mean'),
    barca_avg=('barca_goals', 'mean'),
    match_count=('rm_goals', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(decade_stats))
w = 0.35

bars1 = ax.bar(x - w/2, decade_stats['rm_avg'], w, label='Real Madrid', color=RM_COLOR, edgecolor='black')
bars2 = ax.bar(x + w/2, decade_stats['barca_avg'], w, label='Barcelona', color=BARCA_COLOR, edgecolor='black')

ax.set_xlabel('Decade')
ax.set_ylabel('Average Goals Per Match')
ax.set_title('Average Goals Per Match by Decade', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(decade_stats['decade'], rotation=45)
ax.legend(fontsize=12)

# Add match count labels on top
for i, count in enumerate(decade_stats['match_count']):
    ax.text(i, max(decade_stats['rm_avg'].iloc[i], decade_stats['barca_avg'].iloc[i]) + 0.15,
            f'n={count}', ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

---
## 4. Total Goals Distribution (Histogram + KDE)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(df['total_goals'], bins=range(0, 12), kde=True, color='#2196F3',
             edgecolor='black', alpha=0.7, ax=ax)

mean_val = df['total_goals'].mean()
median_val = df['total_goals'].median()
ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_val:.2f}')
ax.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median = {median_val:.1f}')

ax.set_xlabel('Total Goals in Match')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Total Goals per El Clasico Match', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)

# Annotate skewness and kurtosis
skew = df['total_goals'].skew()
kurt = df['total_goals'].kurtosis()
ax.text(0.95, 0.95, f'Skewness: {skew:.3f}\nKurtosis: {kurt:.3f}',
        transform=ax.transAxes, fontsize=11, va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

---
## 5. Home vs Away Goals (Box Plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot: Home vs Away goals
box_data = pd.DataFrame({
    'Home Goals': df['home_goals'],
    'Away Goals': df['away_goals']
})
box_melted = box_data.melt(var_name='Type', value_name='Goals')
sns.boxplot(data=box_melted, x='Type', y='Goals', hue='Type', palette=['#4CAF50', '#FF5722'], legend=False, ax=axes[0])
axes[0].set_title('Home vs Away Goals Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Goals')

# Box plot: RM vs Barca goals
team_data = pd.DataFrame({
    'Real Madrid': df['rm_goals'],
    'Barcelona': df['barca_goals']
})
team_melted = team_data.melt(var_name='Team', value_name='Goals')
sns.boxplot(data=team_melted, x='Team', y='Goals', hue='Team', palette=[RM_COLOR, BARCA_COLOR], legend=False, ax=axes[1])
axes[1].set_title('Real Madrid vs Barcelona Goals Distribution', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Goals')

plt.tight_layout()
plt.show()

---
## 6. RM Goals vs Barca Goals (Scatter Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

color_list = df['rm_result'].map({'RM Win': RM_COLOR, 'RM Loss': BARCA_COLOR, 'Draw': DRAW_COLOR}).fillna('#999')

ax.scatter(df['rm_goals'], df['barca_goals'], c=color_list, s=80, alpha=0.6, edgecolors='black', linewidth=0.5)

# Diagonal line (draw line)
ax.plot([0, 8], [0, 8], 'k--', alpha=0.3, label='Draw line')

ax.set_xlabel('Real Madrid Goals', fontsize=13)
ax.set_ylabel('Barcelona Goals', fontsize=13)
ax.set_title('Real Madrid vs Barcelona: Goals Scored', fontsize=15, fontweight='bold')
ax.set_aspect('equal')

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=RM_COLOR, markersize=12, label='RM Win'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=BARCA_COLOR, markersize=12, label='Barca Win'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=DRAW_COLOR, markersize=12, label='Draw'),
]
ax.legend(handles=legend_elements, fontsize=12)

plt.tight_layout()
plt.show()

---
## 7. Win Rate by Decade (Stacked Bar Chart)

In [ ]:
decade_results = df.dropna(subset=['decade', 'rm_result']).groupby(['decade', 'rm_result']).size().unstack(fill_value=0)

# Convert to percentages
decade_pct = decade_results.div(decade_results.sum(axis=1), axis=0) * 100

# Reorder columns
col_order = ['RM Win', 'Draw', 'RM Loss']
col_order = [c for c in col_order if c in decade_pct.columns]
decade_pct = decade_pct[col_order]

fig, ax = plt.subplots(figsize=(14, 7))
decade_pct.plot(kind='bar', stacked=True, color=[RM_COLOR, DRAW_COLOR, BARCA_COLOR], ax=ax, edgecolor='black')

ax.set_xlabel('Decade')
ax.set_ylabel('Percentage (%)')
ax.set_title('Win Rate by Decade (Real Madrid Perspective)', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)

# Add total match count on bars
totals = decade_results.sum(axis=1)
for i, total in enumerate(totals):
    ax.text(i, 102, f'n={total}', ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

---
## 8. Home Advantage Analysis (Grouped Bar)

In [ ]:
home_results = df.dropna(subset=['result']).groupby(['home_team', 'result']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart: wins at home vs away for each team
for idx, team in enumerate(['Real Madrid', 'Barcelona']):
    team_home = df[(df['home_team'] == team) & df['result'].notna()]
    team_away = df[(df['away_team'] == team) & df['result'].notna()]

    home_win = len(team_home[team_home['result'] == 'H'])
    home_draw = len(team_home[team_home['result'] == 'D'])
    home_loss = len(team_home[team_home['result'] == 'A'])

    away_win = len(team_away[team_away['result'] == 'A'])
    away_draw = len(team_away[team_away['result'] == 'D'])
    away_loss = len(team_away[team_away['result'] == 'H'])

    color = RM_COLOR if team == 'Real Madrid' else BARCA_COLOR

    x = np.arange(3)
    w = 0.35
    axes[idx].bar(x - w/2, [home_win, home_draw, home_loss], w, label='Home', color=color, alpha=0.9, edgecolor='black')
    axes[idx].bar(x + w/2, [away_win, away_draw, away_loss], w, label='Away', color=color, alpha=0.4, edgecolor='black')

    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(['Win', 'Draw', 'Loss'])
    axes[idx].set_title(f'{team}: Home vs Away Record', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel('Number of Matches')
    axes[idx].legend(fontsize=11)

    # Add value labels
    for bar_group in [axes[idx].containers[0], axes[idx].containers[1]]:
        axes[idx].bar_label(bar_group, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 9. Monthly Match Distribution (Bar Chart)

In [ ]:
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

month_counts = df['month'].dropna().astype(int).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar([month_names.get(m, m) for m in month_counts.index],
              month_counts.values, color='#2196F3', edgecolor='black')

ax.bar_label(bars, fontsize=11, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Matches')
ax.set_title('El Clasico Matches by Month', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 10. Shots vs Goals Correlation (Scatter + Regression)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Home shots vs home goals
sns.regplot(x='home_shots', y='home_goals', data=df, ax=axes[0],
            scatter_kws={'alpha': 0.5, 'color': '#4CAF50'},
            line_kws={'color': 'red'})
corr_h = df['home_shots'].corr(df['home_goals'])
axes[0].set_title(f'Home Shots vs Home Goals (r = {corr_h:.3f})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Home Shots')
axes[0].set_ylabel('Home Goals')

# Total shots vs total goals
sns.regplot(x='total_shots', y='total_goals', data=df, ax=axes[1],
            scatter_kws={'alpha': 0.5, 'color': '#FF9800'},
            line_kws={'color': 'red'})
corr_t = df['total_shots'].corr(df['total_goals'])
axes[1].set_title(f'Total Shots vs Total Goals (r = {corr_t:.3f})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Total Shots')
axes[1].set_ylabel('Total Goals')

plt.tight_layout()
plt.show()

---
## 11. Fouls and Cards Per Match Over Time (Line Chart)

In [ ]:
time_df = df.dropna(subset=['decade']).copy()
decade_agg = time_df.groupby('decade').agg(
    avg_fouls=('total_fouls', 'mean'),
    avg_cards=('total_cards', 'mean'),
    avg_goals=('total_goals', 'mean'),
).reset_index()

fig, ax1 = plt.subplots(figsize=(13, 6))

ax1.plot(decade_agg['decade'], decade_agg['avg_fouls'], 'o-', color='#F44336',
         linewidth=2, markersize=8, label='Avg Fouls/Match')
ax1.set_xlabel('Decade')
ax1.set_ylabel('Average Fouls', color='#F44336')
ax1.tick_params(axis='y', labelcolor='#F44336')

ax2 = ax1.twinx()
ax2.plot(decade_agg['decade'], decade_agg['avg_cards'], 's--', color='#FF9800',
         linewidth=2, markersize=8, label='Avg Cards/Match')
ax2.set_ylabel('Average Cards', color='#FF9800')
ax2.tick_params(axis='y', labelcolor='#FF9800')

fig.suptitle('Fouls and Cards Over Time in El Clasico', fontsize=15, fontweight='bold')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88), fontsize=11)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 12. Goal Category Breakdown (Donut Chart)

In [ ]:
cat_counts = df['goal_category'].value_counts()
cat_order = ['Goalless', 'Low', 'Medium', 'High']
cat_counts = cat_counts.reindex(cat_order).dropna()

colors_cat = ['#9E9E9E', '#4CAF50', '#2196F3', '#F44336']

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    cat_counts.values, labels=cat_counts.index, colors=colors_cat,
    autopct='%1.1f%%', startangle=90, pctdistance=0.82,
    textprops={'fontsize': 13, 'fontweight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2, 'width': 0.4}
)
for t in autotexts:
    t.set_fontsize(11)

ax.set_title('Match Scoring Categories', fontsize=16, fontweight='bold')

# Center text
ax.text(0, 0, f'{len(df)}\nMatches', ha='center', va='center', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 13. Biggest Wins (Top 10) (Horizontal Bar)

In [ ]:
df_sorted = df.dropna(subset=['date_str']).copy()
df_sorted['abs_goal_diff'] = df_sorted['goal_diff'].abs()
top10 = df_sorted.nlargest(10, 'abs_goal_diff').sort_values('abs_goal_diff')

labels = []
colors_bar = []
for _, row in top10.iterrows():
    h, a = int(row['home_goals']), int(row['away_goals'])
    labels.append(f"{row['home_team']} {h}-{a} {row['away_team']}  ({row['date_str'][:4]})")
    if row['winner'] == 'Real Madrid':
        colors_bar.append(RM_COLOR)
    elif row['winner'] == 'Barcelona':
        colors_bar.append(BARCA_COLOR)
    else:
        colors_bar.append(DRAW_COLOR)

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(labels, top10['abs_goal_diff'], color=colors_bar, edgecolor='black')
ax.set_xlabel('Goal Difference')
ax.set_title('Top 10 Biggest Wins in El Clasico History', fontsize=15, fontweight='bold')

for i, v in enumerate(top10['abs_goal_diff']):
    ax.text(v + 0.1, i, f'{int(v)}', va='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

---
## 14. Correlation Matrix (Heatmap)

In [ ]:
corr_cols = ['home_goals', 'away_goals', 'total_goals',
             'home_shots', 'away_shots', 'home_shots_on_target', 'away_shots_on_target',
             'home_fouls', 'away_fouls', 'home_corners', 'away_corners',
             'home_yellows', 'away_yellows', 'home_reds', 'away_reds']

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            annot_kws={'fontsize': 9})
ax.set_title('Correlation Matrix: Match Statistics', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 15. Corners Comparison (Violin Plot)

In [ ]:
corners_df = pd.melt(
    df[['home_corners', 'away_corners', 'home_team']],
    id_vars=['home_team'],
    value_vars=['home_corners', 'away_corners'],
    var_name='type', value_name='corners'
)
corners_df['type'] = corners_df['type'].map({'home_corners': 'Home Team', 'away_corners': 'Away Team'})

fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(x='type', y='corners', data=corners_df, hue='type',
               palette=['#4CAF50', '#FF5722'], inner='quart', legend=False, ax=ax)
ax.set_title('Corner Kicks Distribution: Home vs Away', fontsize=15, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Corners')

plt.tight_layout()
plt.show()

---
## 16. Half-Time vs Full-Time Results (Grouped Bar)

In [ ]:
ht_ft = df.dropna(subset=['ht_result', 'result']).copy()

ht_counts = ht_ft['ht_result'].value_counts().reindex(['H', 'D', 'A'], fill_value=0)
ft_counts = ht_ft['result'].value_counts().reindex(['H', 'D', 'A'], fill_value=0)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(3)
w = 0.35

ax.bar(x - w/2, ht_counts.values, w, label='Half-Time', color='#42A5F5', edgecolor='black')
ax.bar(x + w/2, ft_counts.values, w, label='Full-Time', color='#1565C0', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(['Home Win', 'Draw', 'Away Win'])
ax.set_ylabel('Number of Matches')
ax.set_title('Half-Time vs Full-Time Results', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)

for container in ax.containers:
    ax.bar_label(container, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 17. Shots on Target Efficiency (Bar Chart)

In [ ]:
# Calculate shot conversion rate when team is at home vs away
rm_home = df[df['home_team'] == 'Real Madrid']
rm_away = df[df['away_team'] == 'Real Madrid']
barca_home = df[df['home_team'] == 'Barcelona']
barca_away = df[df['away_team'] == 'Barcelona']

data = {
    'Team': ['RM (Home)', 'RM (Away)', 'Barca (Home)', 'Barca (Away)'],
    'Avg Shots': [
        rm_home['home_shots'].mean(),
        rm_away['away_shots'].mean(),
        barca_home['home_shots'].mean(),
        barca_away['away_shots'].mean(),
    ],
    'Avg SOT': [
        rm_home['home_shots_on_target'].mean(),
        rm_away['away_shots_on_target'].mean(),
        barca_home['home_shots_on_target'].mean(),
        barca_away['away_shots_on_target'].mean(),
    ],
    'Avg Goals': [
        rm_home['home_goals'].mean(),
        rm_away['away_goals'].mean(),
        barca_home['home_goals'].mean(),
        barca_away['away_goals'].mean(),
    ],
}
eff_df = pd.DataFrame(data)
eff_df['SOT %'] = (eff_df['Avg SOT'] / eff_df['Avg Shots'] * 100).round(1)
eff_df['Conversion %'] = (eff_df['Avg Goals'] / eff_df['Avg Shots'] * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(4)
w = 0.25

ax.bar(x - w, eff_df['Avg Shots'], w, label='Shots', color='#90CAF9', edgecolor='black')
ax.bar(x, eff_df['Avg SOT'], w, label='Shots on Target', color='#42A5F5', edgecolor='black')
ax.bar(x + w, eff_df['Avg Goals'], w, label='Goals', color='#1565C0', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(eff_df['Team'])
ax.set_ylabel('Average Per Match')
ax.set_title('Shooting Efficiency: Shots -> On Target -> Goals', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)

# Add conversion % labels
for i, row in eff_df.iterrows():
    ax.text(i + w, row['Avg Goals'] + 0.2, f"{row['Conversion %']}%", ha='center', fontsize=10, color='#1565C0', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 18. Outlier Matches (Strip Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

normal = df[~df['is_outlier_zscore']]
outliers = df[df['is_outlier_zscore']]

ax.scatter(normal['year'], normal['total_goals'], c='#90CAF9', s=50, alpha=0.6, label='Normal', edgecolors='gray')
ax.scatter(outliers['year'], outliers['total_goals'], c='#F44336', s=120, alpha=0.9,
           label='Outlier (Z > 2.5)', edgecolors='black', linewidth=1.5, zorder=5)

# Annotate outliers
for _, row in outliers.iterrows():
    h, a = int(row['home_goals']), int(row['away_goals'])
    ax.annotate(f"{h}-{a}", xy=(row['year'], row['total_goals']),
                xytext=(5, 10), textcoords='offset points',
                fontsize=10, fontweight='bold', color='#F44336')

ax.set_xlabel('Year')
ax.set_ylabel('Total Goals')
ax.set_title('Outlier Matches by Total Goals (Z-Score Method)', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

---
## 19. Cumulative Wins Over Time (Line Chart)

In [ ]:
timeline = df.dropna(subset=['year', 'rm_result']).sort_values('year').copy()
timeline['rm_cum_wins'] = (timeline['rm_result'] == 'RM Win').cumsum()
timeline['barca_cum_wins'] = (timeline['rm_result'] == 'RM Loss').cumsum()
timeline['draw_cum'] = (timeline['rm_result'] == 'Draw').cumsum()

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(timeline['year'], timeline['rm_cum_wins'], linewidth=2.5, color=RM_COLOR, label='Real Madrid Wins')
ax.plot(timeline['year'], timeline['barca_cum_wins'], linewidth=2.5, color=BARCA_COLOR, label='Barcelona Wins')
ax.plot(timeline['year'], timeline['draw_cum'], linewidth=1.5, color=DRAW_COLOR, linestyle='--', label='Draws')

ax.fill_between(timeline['year'], timeline['rm_cum_wins'], timeline['barca_cum_wins'],
                where=timeline['rm_cum_wins'] >= timeline['barca_cum_wins'],
                alpha=0.15, color=RM_COLOR)
ax.fill_between(timeline['year'], timeline['rm_cum_wins'], timeline['barca_cum_wins'],
                where=timeline['rm_cum_wins'] < timeline['barca_cum_wins'],
                alpha=0.15, color=BARCA_COLOR)

ax.set_xlabel('Year')
ax.set_ylabel('Cumulative Wins')
ax.set_title('Cumulative Wins Over Time: Real Madrid vs Barcelona', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

---
## 20. K-Means Cluster Visualization (Scatter Plot)

In [ ]:
cluster_df = df.dropna(subset=['kmeans_cluster']).copy()
cluster_df['kmeans_cluster'] = cluster_df['kmeans_cluster'].astype(int)

cluster_colors = {0: '#2196F3', 1: '#4CAF50', 2: '#FF9800'}
cluster_labels = {0: 'Low-scoring', 1: 'Balanced', 2: 'High-scoring'}

# Rename clusters based on avg total goals
cluster_means = cluster_df.groupby('kmeans_cluster')['total_goals'].mean().sort_values()
rename = {cluster_means.index[0]: 'Low-scoring',
          cluster_means.index[1]: 'Balanced',
          cluster_means.index[2]: 'High-scoring'}
cluster_df['cluster_name'] = cluster_df['kmeans_cluster'].map(rename)

fig, ax = plt.subplots(figsize=(10, 8))

for name, group in cluster_df.groupby('cluster_name'):
    ax.scatter(group['home_goals'], group['away_goals'], s=80, alpha=0.7,
               edgecolors='black', linewidth=0.5, label=f'{name} (n={len(group)})')

# Mark outliers
km_outliers = cluster_df[cluster_df['is_outlier_kmeans']]
ax.scatter(km_outliers['home_goals'], km_outliers['away_goals'],
           s=200, facecolors='none', edgecolors='red', linewidths=2, label='Outlier')

ax.plot([0, 8], [0, 8], 'k--', alpha=0.2)
ax.set_xlabel('Home Goals', fontsize=13)
ax.set_ylabel('Away Goals', fontsize=13)
ax.set_title('K-Means Clustering of El Clasico Matches', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

---
## Summary Statistics Table

In [ ]:
print('EL CLASICO | KEY STATISTICS')
print('=' * 50)
print(f'Total matches analyzed       : {len(df)}')
print(f'Date range                   : {df["year"].min():.0f} - {df["year"].max():.0f}')
print(f'Total goals scored           : {df["total_goals"].sum():.0f}')
print(f'Average goals per match      : {df["total_goals"].mean():.2f}')
print(f'Highest scoring match        : {df["total_goals"].max():.0f} goals')
print(f'Goalless draws               : {len(df[df["total_goals"]==0])}')
print()

rm_wins = (df['rm_result'] == 'RM Win').sum()
barca_wins = (df['rm_result'] == 'RM Loss').sum()
draws = (df['rm_result'] == 'Draw').sum()

print(f'Real Madrid wins             : {rm_wins} ({rm_wins/len(df)*100:.1f}%)')
print(f'Barcelona wins               : {barca_wins} ({barca_wins/len(df)*100:.1f}%)')
print(f'Draws                        : {draws} ({draws/len(df)*100:.1f}%)')
print()
print(f'RM total goals               : {df["rm_goals"].sum():.0f} (avg {df["rm_goals"].mean():.2f}/match)')
print(f'Barca total goals            : {df["barca_goals"].sum():.0f} (avg {df["barca_goals"].mean():.2f}/match)')
print()
print(f'Outlier matches (Z-score)    : {df["is_outlier_zscore"].sum()}')
print(f'Outlier matches (IQR)        : {df["is_outlier_iqr"].sum()}')
print(f'Outlier matches (K-Means)    : {df["is_outlier_kmeans"].sum()}')